# 14 — Walidacja syntetyczna (T12)


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

D = 5            # liczba cech kontekstu
PI_EVAL = 0.5    # eval policy: P(T=1) = 50% ("aggressive", jak w T11)
GAMMA_0 = -4.5   # baseline CTR ~1.8% (rzędu wielkości StatsBomb: 1.84%)
N = 20000
N_BOOTSTRAP = 200
RANDOM_STATE = 42

FIGURES_DIR = Path("../figures/week12")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("../results/week12")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


## Generator (znane V*, V_naive)


In [2]:
def make_data(n, gamma_t, w_b_scale, gamma_0=GAMMA_0, w_r_scale=0.3, alpha_b=-0.55, rng=None):
    s = rng.normal(size=(n, D))
    w_r = rng.normal(scale=w_r_scale, size=D)
    w_b = rng.normal(scale=w_b_scale, size=D)

    score_r = s @ w_r
    p0 = sigmoid(gamma_0 + score_r)              # E[Y | X, T=0]
    p1 = sigmoid(gamma_0 + gamma_t + score_r)    # E[Y | X, T=1]
    pb = sigmoid(alpha_b + s @ w_b)              # P_b(T=1 | X) -- prawdziwa logging policy

    T = (rng.uniform(size=n) < pb).astype(int)
    p_obs = np.where(T == 1, p1, p0)
    Y = (rng.uniform(size=n) < p_obs).astype(float)

    return dict(s=s, T=T, Y=Y, p0=p0, p1=p1, pb=pb)


def true_values(data):
    p0, p1, pb = data["p0"], data["p1"], data["pb"]
    v_star = float(np.mean(PI_EVAL * p1 + (1 - PI_EVAL) * p0))
    v_naive = float(np.mean(pb * p1 + (1 - pb) * p0))
    return v_star, v_naive


## Pipeline OPE


In [3]:
def run_ope(data, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_STATE):
    s, T, Y = data["s"], data["T"], data["Y"]
    n = len(Y)

    # Propensity model (jak w T11)
    ps_model = LogisticRegression(max_iter=500, random_state=seed, C=1.0)
    ps_model.fit(s, T)
    pscore = np.clip(ps_model.predict_proba(s)[:, 1], 0.01, 0.99)

    # Reward model (jak w T11)
    X_full = np.hstack([s, T.reshape(-1, 1)])
    X_t1 = np.hstack([s, np.ones((n, 1))])
    X_t0 = np.hstack([s, np.zeros((n, 1))])
    reward_model = XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.05,
        subsample=0.8, eval_metric="logloss",
        tree_method="hist", n_jobs=1, random_state=seed,
    )
    reward_model.fit(X_full, Y, verbose=False)
    er_obs = reward_model.predict_proba(X_full)[:, 1]
    er_t1 = reward_model.predict_proba(X_t1)[:, 1]
    er_t0 = reward_model.predict_proba(X_t0)[:, 1]
    er_eval = PI_EVAL * er_t1 + (1 - PI_EVAL) * er_t0

    weights = np.where(T == 1, PI_EVAL / pscore, (1 - PI_EVAL) / (1 - pscore))
    ess = float(weights.sum() ** 2 / (weights ** 2).sum()) / n

    def dm_fn(er_eval, **_):
        return float(np.mean(er_eval))

    def ips_fn(Y, weights, **_):
        return float(np.mean(weights * Y))

    def snips_fn(Y, weights, **_):
        return float(np.sum(weights * Y) / np.sum(weights))

    def dr_fn(Y, weights, er_eval, er_obs, **_):
        return float(np.mean(weights * (Y - er_obs)) + np.mean(er_eval))

    fns = {"DM": dm_fn, "IPS": ips_fn, "SNIPS": snips_fn, "DR": dr_fn}
    point = {
        "DM": dm_fn(er_eval=er_eval),
        "IPS": ips_fn(Y=Y, weights=weights),
        "SNIPS": snips_fn(Y=Y, weights=weights),
        "DR": dr_fn(Y=Y, weights=weights, er_eval=er_eval, er_obs=er_obs),
    }

    rng = np.random.default_rng(seed)
    boots = {k: [] for k in point}
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        boots["DM"].append(dm_fn(er_eval=er_eval[idx]))
        boots["IPS"].append(ips_fn(Y=Y[idx], weights=weights[idx]))
        boots["SNIPS"].append(snips_fn(Y=Y[idx], weights=weights[idx]))
        boots["DR"].append(dr_fn(Y=Y[idx], weights=weights[idx], er_eval=er_eval[idx], er_obs=er_obs[idx]))

    ci = {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in boots.items()}
    return point, ci, ess, weights


## Oracle check


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
data_oracle = make_data(N, gamma_t=2.0, w_b_scale=0.55, rng=rng)
v_star_o, v_naive_o = true_values(data_oracle)
s, T, Y, p0, p1, pb = (data_oracle[k] for k in ["s", "T", "Y", "p0", "p1", "pb"])

w_oracle = np.where(T == 1, PI_EVAL / pb, (1 - PI_EVAL) / (1 - pb))
er_obs_oracle = np.where(T == 1, p1, p0)
er_eval_oracle = PI_EVAL * p1 + (1 - PI_EVAL) * p0

oracle = {
    "DM":    float(er_eval_oracle.mean()),
    "IPS":   float((w_oracle * Y).mean()),
    "SNIPS": float((w_oracle * Y).sum() / w_oracle.sum()),
    "DR":    float((w_oracle * (Y - er_obs_oracle)).mean() + er_eval_oracle.mean()),
}

print(f"V* = {v_star_o:.5f}   V_naive = {v_naive_o:.5f}")
for k, v in oracle.items():
    print(f"  oracle {k:6s} = {v:.5f}   bias vs V* = {v - v_star_o:+.5f}")


## Scenariusze A–E


In [ ]:
scenarios = {
    "A. Dobry overlap, Delta=0":      dict(gamma_t=0.0, w_b_scale=0.05),
    "B. Dobry overlap, Delta=duzy":   dict(gamma_t=2.0, w_b_scale=0.05),
    "C. StatsBomb overlap, Delta=0":      dict(gamma_t=0.0, w_b_scale=0.55),
    "D. StatsBomb overlap, Delta=maly":   dict(gamma_t=0.8, w_b_scale=0.55),
    "E. StatsBomb overlap, Delta=duzy":   dict(gamma_t=2.0, w_b_scale=0.55),
}

rows = []
raw = {}
for name, params in scenarios.items():
    rng = np.random.default_rng(RANDOM_STATE)
    data = make_data(N, rng=rng, **params)
    v_star, v_naive = true_values(data)
    point, ci, ess, weights = run_ope(data)
    raw[name] = dict(v_star=v_star, v_naive=v_naive, ess=ess, point=point, ci=ci)
    for est in ["DM", "IPS", "SNIPS", "DR"]:
        lo, hi = ci[est]
        rows.append({
            "Scenario": name,
            "Estimator": est,
            "V*": v_star,
            "V_naive": v_naive,
            "true_effect": v_star - v_naive,
            "ESS": ess,
            "V_hat": point[est],
            "CI_lower": lo,
            "CI_upper": hi,
            "CI_width": hi - lo,
            "bias_vs_Vstar": point[est] - v_star,
            "naive_in_CI": lo <= v_naive <= hi,
            "Vstar_in_CI": lo <= v_star <= hi,
        })

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_DIR / "synthetic_validation.csv", index=False)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
results_df


## Wizualizacja


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5), sharey=False)
colors_e = {"DM": "steelblue", "IPS": "darkorange", "SNIPS": "seagreen", "DR": "mediumpurple"}

for ax, (name, info) in zip(axes, raw.items()):
    ests = ["DM", "IPS", "SNIPS", "DR"]
    means = [info["point"][e] for e in ests]
    los = [info["ci"][e][0] for e in ests]
    his = [info["ci"][e][1] for e in ests]
    err = [[m - lo for m, lo in zip(means, los)], [hi - m for m, hi in zip(means, his)]]

    ax.bar(ests, means, yerr=err, color=[colors_e[e] for e in ests], alpha=0.85, capsize=5)
    ax.axhline(info["v_star"], color="red", linestyle="--", linewidth=1.5, label="V*")
    ax.axhline(info["v_naive"], color="gray", linestyle=":", linewidth=1.5, label="V_naive")
    ax.set_title(name.split(". ")[1] + f"\nESS={info['ess']:.2f}, true_effect={info['v_star']-info['v_naive']:+.4f}", fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle("Walidacja syntetyczna: estymaty (95% CI) vs V* (prawda, czerwona) i V_naive (baseline, szara)", fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "synthetic_validation.png", dpi=160, bbox_inches="tight")
plt.show()
